# Day 14 — Deep Learning Robustness Review and Candidate Decision

**Project:** IPL Player Performance Analytics and Prediction Using Machine Learning and Deep Learning

## Objectives
- Review the Day 12 LSTM/GRU results and Day 13 ML-vs-DL comparison.
- Analyse selected deep-learning test residuals and identify large errors.
- Check prediction bias and error patterns by target and available player metadata.
- Verify candidate artifacts and feature manifests.
- Record whether each deep-learning target is ready to carry forward, needs review, or is blocked.

**Important:** this notebook does not retrain models, tune using the test set, or invent results. Test residuals are diagnostic only and must not be used to select a different architecture.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

current_dir = Path.cwd().resolve()
candidates = [current_dir, *current_dir.parents]
def is_project_root(p):
    return (p / "reports" / "day12").is_dir() or (
        (p / "reports").is_dir() and (p / "data" / "processed").is_dir()
    )
PROJECT_ROOT = next((p for p in candidates if is_project_root(p)), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path(r"D:\Git\Personal\ipl-player-performance")

REPORT_DIR = PROJECT_ROOT / "reports"
DAY12_DIR = REPORT_DIR / "day12"
DAY13_DIR = REPORT_DIR / "day13"
DAY14_DIR = REPORT_DIR / "day14"
MODEL_DIR = PROJECT_ROOT / "models" / "day12"
DOCS_DIR = PROJECT_ROOT / "docs"
for folder in [DAY14_DIR, DOCS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

PATHS = {
    "day12_metrics": DAY12_DIR / "selected_lstm_gru_test_metrics.csv",
    "day12_predictions": DAY12_DIR / "selected_lstm_gru_test_predictions.csv",
    "day12_manifest": DAY12_DIR / "model_artifacts_manifest.csv",
    "day12_status": DAY12_DIR / "task_status.csv",
    "day13_comparison": DAY13_DIR / "ml_vs_dl_model_comparison.csv",
    "day13_interpretation": DAY13_DIR / "comparison_interpretation.csv",
}
print("Project root:", PROJECT_ROOT)
for name, path in PATHS.items():
    print(("FOUND  " if path.is_file() else "MISSING"), name, "->", path)

Project root: D:\Git\Personal\ipl-player-performance
FOUND   day12_metrics -> D:\Git\Personal\ipl-player-performance\reports\day12\selected_lstm_gru_test_metrics.csv
FOUND   day12_predictions -> D:\Git\Personal\ipl-player-performance\reports\day12\selected_lstm_gru_test_predictions.csv
FOUND   day12_manifest -> D:\Git\Personal\ipl-player-performance\reports\day12\model_artifacts_manifest.csv
FOUND   day12_status -> D:\Git\Personal\ipl-player-performance\reports\day12\task_status.csv
FOUND   day13_comparison -> D:\Git\Personal\ipl-player-performance\reports\day13\ml_vs_dl_model_comparison.csv
FOUND   day13_interpretation -> D:\Git\Personal\ipl-player-performance\reports\day13\comparison_interpretation.csv


## 1. Load Day 12/13 outputs and verify schemas

The notebook proceeds with whichever reports are available but marks missing dependencies. The prediction file should include actual, predicted, residual, target, dataset, and selected architecture columns.

In [2]:
def read_optional(path):
    return pd.read_csv(path) if path.is_file() else None

frames = {name: read_optional(path) for name, path in PATHS.items()}
source_status = pd.DataFrame([
    {"source": name, "status": "FOUND" if frame is not None else "MISSING",
     "path": str(PATHS[name]),
     "rows": len(frame) if frame is not None else 0}
    for name, frame in frames.items()
])
display(source_status)

predictions = frames["day12_predictions"]
required_prediction_cols = {
    "target", "actual", "predicted", "residual_actual_minus_predicted"
}
if predictions is not None:
    missing_cols = required_prediction_cols - set(predictions.columns)
    print("Prediction columns:", list(predictions.columns))
    print("Missing required prediction columns:", sorted(missing_cols))
else:
    print("Day 12 prediction report is missing; residual analysis cannot run.")

,source,status,path,rows
0,day12_metrics,FOUND,D:\Git\Personal\ipl-player-performance\reports...,3
1,day12_predictions,FOUND,D:\Git\Personal\ipl-player-performance\reports...,6076
2,day12_manifest,FOUND,D:\Git\Personal\ipl-player-performance\reports...,3
3,day12_status,FOUND,D:\Git\Personal\ipl-player-performance\reports...,3
4,day13_comparison,FOUND,D:\Git\Personal\ipl-player-performance\reports...,6
5,day13_interpretation,FOUND,D:\Git\Personal\ipl-player-performance\reports...,3


Prediction columns: ['dataset', 'target', 'player_name', 'history_start_match_id', 'history_end_match_id', 'history_end_date', 'target_match_date', 'target_history_row_index', 'history_matches', 'selected_architecture', 'actual', 'predicted', 'residual_actual_minus_predicted']
Missing required prediction columns: []


## 2. Residual and error diagnostics

Residual is defined as `actual - predicted`. Positive mean residual indicates under-prediction on average; negative mean residual indicates over-prediction. The 90th/95th percentile absolute error and largest errors help identify cases worth checking against source data and target alignment.

In [3]:
error_summary_rows = []
largest_error_frames = []
group_summary_rows = []

if predictions is not None and required_prediction_cols.issubset(predictions.columns):
    pred = predictions.copy()
    for col in ["actual", "predicted", "residual_actual_minus_predicted"]:
        pred[col] = pd.to_numeric(pred[col], errors="coerce")
    pred["absolute_error"] = pred["residual_actual_minus_predicted"].abs()
    pred = pred.dropna(subset=["actual", "predicted", "residual_actual_minus_predicted"])

    for target, group in pred.groupby("target", dropna=False):
        if group.empty:
            continue
        abs_error = group["absolute_error"]
        error_summary_rows.append({
            "target": target,
            "rows": len(group),
            "MAE_recomputed": float(abs_error.mean()),
            "RMSE_recomputed": float(np.sqrt(np.mean(group["residual_actual_minus_predicted"] ** 2))),
            "mean_residual_actual_minus_predicted": float(group["residual_actual_minus_predicted"].mean()),
            "median_absolute_error": float(abs_error.median()),
            "p90_absolute_error": float(abs_error.quantile(0.90)),
            "p95_absolute_error": float(abs_error.quantile(0.95)),
            "max_absolute_error": float(abs_error.max()),
        })
        largest = group.nlargest(min(20, len(group)), "absolute_error").copy()
        keep_cols = [c for c in [
            "dataset", "target", "player_name", "history_end_match_id",
            "target_match_date", "actual", "predicted",
            "residual_actual_minus_predicted", "absolute_error", "selected_architecture"
        ] if c in largest.columns]
        largest_error_frames.append(largest[keep_cols])

        for col in ["player_name", "selected_architecture"]:
            if col in group.columns:
                for key, sub in group.groupby(col, dropna=False):
                    group_summary_rows.append({
                        "target": target, "group_column": col, "group_value": str(key),
                        "rows": len(sub),
                        "MAE": float(sub["absolute_error"].mean()),
                        "RMSE": float(np.sqrt(np.mean(sub["residual_actual_minus_predicted"] ** 2))),
                        "mean_residual": float(sub["residual_actual_minus_predicted"].mean())
                    })

error_summary_df = pd.DataFrame(error_summary_rows)
largest_errors_df = pd.concat(largest_error_frames, ignore_index=True) if largest_error_frames else pd.DataFrame()
group_summary_df = pd.DataFrame(group_summary_rows)
display(error_summary_df)
display(largest_errors_df.head(30))

,target,rows,MAE_recomputed,RMSE_recomputed,mean_residual_actual_minus_predicted,median_absolute_error,p90_absolute_error,p95_absolute_error,max_absolute_error
0,next_match_runs,2374,16.706737,22.493525,2.379298,13.352062,34.245324,45.780291,119.059250
1,next_match_runs_conceded,1851,9.456312,11.845119,3.307186,7.983307,19.361038,23.433829,46.539734
2,next_match_wickets,1851,0.790386,1.000189,-0.007816,0.850177,1.325027,1.942337,4.040541


,dataset,target,player_name,history_end_match_id,target_match_date,actual,predicted,residual_actual_minus_predicted,absolute_error,selected_architecture
0,batting,next_match_runs,Abhishek Sharma,1473457,2025-04-12,141.0,21.940752,119.059250,119.059250,LSTM
1,batting,next_match_runs,KL Rahul,1529274,2026-04-25,152.0,33.301840,118.698160,118.698160,LSTM
2,batting,next_match_runs,Abhishek Sharma,1529270,2026-04-21,135.0,22.461483,112.538510,112.538510,LSTM
3,batting,next_match_runs,RD Rickelton,1529267,2026-04-29,123.0,25.437464,97.562540,97.562540,LSTM
4,batting,next_match_runs,RR Pant,1473502,2025-05-27,118.0,24.103302,93.896700,93.896700,LSTM
5,batting,next_match_runs,MR Marsh,1473499,2025-05-22,117.0,24.053062,92.946940,92.946940,LSTM
6,batting,next_match_runs,V Suryavanshi,1529275,2026-04-25,103.0,10.820400,92.179596,92.179596,LSTM
7,batting,next_match_runs,SV Samson,1527684,2026-04-11,115.0,23.999220,91.000780,91.000780,LSTM
8,batting,next_match_runs,Q de Kock,1473505,2026-04-16,112.0,23.747484,88.252520,88.252520,LSTM
9,batting,next_match_runs,FH Allen,1529288,2026-05-08,100.0,11.900501,88.099500,88.099500,LSTM


## 3. Compare recomputed metrics to the Day 12 metric report

This is a consistency check, not model selection. Small floating-point differences may occur; larger discrepancies should be investigated.

In [4]:
metric_consistency_rows = []
day12_metrics = frames["day12_metrics"]
if day12_metrics is not None and not error_summary_df.empty and "target" in day12_metrics.columns:
    for _, row in error_summary_df.iterrows():
        match = day12_metrics[day12_metrics["target"].astype(str) == str(row["target"])]
        if match.empty:
            metric_consistency_rows.append({
                "target": row["target"], "status": "NO MATCH IN DAY12 METRICS",
                "recomputed_MAE": row["MAE_recomputed"], "reported_MAE": np.nan,
                "recomputed_RMSE": row["RMSE_recomputed"], "reported_RMSE": np.nan
            })
            continue
        reported = match.iloc[0]
        mae_col = next((c for c in ["test_MAE", "MAE", "test_mae"] if c in match.columns), None)
        rmse_col = next((c for c in ["test_RMSE", "RMSE", "test_rmse"] if c in match.columns), None)
        reported_mae = pd.to_numeric(reported[mae_col], errors="coerce") if mae_col else np.nan
        reported_rmse = pd.to_numeric(reported[rmse_col], errors="coerce") if rmse_col else np.nan
        mae_ok = pd.notna(reported_mae) and np.isclose(row["MAE_recomputed"], reported_mae, rtol=1e-3, atol=1e-4)
        rmse_ok = pd.notna(reported_rmse) and np.isclose(row["RMSE_recomputed"], reported_rmse, rtol=1e-3, atol=1e-4)
        metric_consistency_rows.append({
            "target": row["target"],
            "status": "MATCH" if mae_ok and rmse_ok else "REVIEW",
            "recomputed_MAE": row["MAE_recomputed"], "reported_MAE": reported_mae,
            "recomputed_RMSE": row["RMSE_recomputed"], "reported_RMSE": reported_rmse
        })
else:
    print("Metric consistency check skipped: Day 12 metrics or prediction diagnostics are missing.")

metric_consistency_df = pd.DataFrame(metric_consistency_rows)
display(metric_consistency_df)

,target,status,recomputed_MAE,reported_MAE,recomputed_RMSE,reported_RMSE
0,next_match_runs,MATCH,16.706737,16.706738,22.493525,22.493524
1,next_match_runs_conceded,MATCH,9.456312,9.456311,11.845119,11.845119
2,next_match_wickets,MATCH,0.790386,0.790386,1.000189,1.000189


## 4. Check candidate artifacts and model readiness

The goal is to decide whether each target can be carried forward to the integration phase. `READY FOR INTEGRATION` means required artifacts and reports exist; it does not mean the model is scientifically optimal or production-validated.

In [5]:
manifest = frames["day12_manifest"]
artifact_checks = []
if manifest is None:
    artifact_checks.append({
        "target": "ALL", "status": "BLOCKED",
        "detail": "Day 12 model artifacts manifest is missing."
    })
else:
    target_col = "target" if "target" in manifest.columns else None
    model_path_col = "model_path" if "model_path" in manifest.columns else None
    imputer_col = "imputer_path" if "imputer_path" in manifest.columns else None
    scaler_col = "scaler_path" if "scaler_path" in manifest.columns else None
    for _, row in manifest.iterrows():
        target = str(row[target_col]) if target_col else "UNKNOWN"
        required_paths = []
        for col in [model_path_col, imputer_col, scaler_col]:
            if col and pd.notna(row[col]):
                p = Path(str(row[col]))
                if not p.is_absolute():
                    p = PROJECT_ROOT / p
                required_paths.append((col, p))
        missing = [f"{col}: {p}" for col, p in required_paths if not p.is_file()]
        artifact_checks.append({
            "target": target,
            "status": "PASS" if required_paths and not missing else "REVIEW",
            "detail": "All listed artifacts exist." if required_paths and not missing
                      else ("Missing files: " + "; ".join(missing) if missing else "Manifest paths missing or incomplete.")
        })

artifact_checks_df = pd.DataFrame(artifact_checks)
display(artifact_checks_df)

readiness_rows = []
targets = ["next_match_runs", "next_match_wickets", "next_match_runs_conceded"]
for target in targets:
    status = "READY FOR REVIEW"
    reasons = []
    if predictions is None or not required_prediction_cols.issubset(predictions.columns):
        status = "BLOCKED"
        reasons.append("Day 12 test predictions unavailable or schema incomplete.")
    if error_summary_df.empty or target not in set(error_summary_df.get("target", pd.Series(dtype=str)).astype(str)):
        status = "BLOCKED"
        reasons.append("No residual summary for this target.")
    if not artifact_checks_df.empty:
        target_artifacts = artifact_checks_df[
            (artifact_checks_df["target"].astype(str) == target)
            | (artifact_checks_df["target"].astype(str) == "ALL")
        ]
        if target_artifacts.empty or not target_artifacts["status"].eq("PASS").all():
            if status != "BLOCKED":
                status = "REVIEW"
            reasons.append("Candidate artifact checks are incomplete or failed.")
    if not metric_consistency_df.empty:
        target_consistency = metric_consistency_df[metric_consistency_df["target"].astype(str) == target]
        if target_consistency.empty or not target_consistency["status"].eq("MATCH").all():
            if status != "BLOCKED":
                status = "REVIEW"
            reasons.append("Reported and recomputed metrics require review.")
    if status == "READY FOR REVIEW":
        status = "READY FOR INTEGRATION"
        reasons.append("Required reports and artifacts passed the notebook checks.")
    readiness_rows.append({"target": target, "readiness": status, "notes": "; ".join(reasons)})

readiness_df = pd.DataFrame(readiness_rows)
display(readiness_df)

,target,status,detail
0,next_match_runs,PASS,All listed artifacts exist.
1,next_match_wickets,PASS,All listed artifacts exist.
2,next_match_runs_conceded,PASS,All listed artifacts exist.


,target,readiness,notes
0,next_match_runs,READY FOR INTEGRATION,Required reports and artifacts passed the note...
1,next_match_wickets,READY FOR INTEGRATION,Required reports and artifacts passed the note...
2,next_match_runs_conceded,READY FOR INTEGRATION,Required reports and artifacts passed the note...


## 5. Export reports and generate Day 14 documentation

Review the largest errors manually. They may reflect genuine unpredictable performance, limited historical samples, source-data problems, or target alignment issues. Do not delete difficult examples solely to improve reported metrics.

In [6]:
error_summary_df.to_csv(DAY14_DIR / "deep_learning_error_summary.csv", index=False)
largest_errors_df.to_csv(DAY14_DIR / "largest_deep_learning_errors.csv", index=False)
group_summary_df.to_csv(DAY14_DIR / "grouped_error_summary.csv", index=False)
metric_consistency_df.to_csv(DAY14_DIR / "metric_consistency_check.csv", index=False)
artifact_checks_df.to_csv(DAY14_DIR / "artifact_validation.csv", index=False)
readiness_df.to_csv(DAY14_DIR / "candidate_readiness.csv", index=False)
source_status.to_csv(DAY14_DIR / "input_status.csv", index=False)

def frame_text(frame):
    return frame.to_string(index=False) if frame is not None and not frame.empty else "No rows produced."

doc = f"""# Day 14 — Deep Learning Robustness Review

## Input availability
```text
{frame_text(source_status)}
```

## Error summary
```text
{frame_text(error_summary_df)}
```

## Metric consistency
```text
{frame_text(metric_consistency_df)}
```

## Artifact validation
```text
{frame_text(artifact_checks_df)}
```

## Candidate readiness
```text
{frame_text(readiness_df)}
```

## Outputs
- `reports/day14/deep_learning_error_summary.csv`
- `reports/day14/largest_deep_learning_errors.csv`
- `reports/day14/grouped_error_summary.csv`
- `reports/day14/metric_consistency_check.csv`
- `reports/day14/artifact_validation.csv`
- `reports/day14/candidate_readiness.csv`
- `reports/day14/input_status.csv`
- `docs/day14_deep_learning_robustness_review.md`

## Method and limitations
- Residual = actual minus predicted.
- Positive mean residual indicates average under-prediction; negative indicates average over-prediction.
- Recomputed metrics are compared to Day 12 reported metrics for consistency only.
- Test errors are diagnostics, not a basis for choosing a different architecture.
- Artifact presence is not proof of production readiness or scientific validity.
- Row/sample-level chronological splits may place different players from the same match across partitions.
- No error rows are removed automatically.

## Next
Day 15: configure MySQL integration and prepare the project database schema.
"""
doc_path = DOCS_DIR / "day14_deep_learning_robustness_review.md"
doc_path.write_text(doc, encoding="utf-8")
print("Saved documentation:", doc_path)
print("Reports written to:", DAY14_DIR)
display(readiness_df)

Saved documentation: D:\Git\Personal\ipl-player-performance\docs\day14_deep_learning_robustness_review.md
Reports written to: D:\Git\Personal\ipl-player-performance\reports\day14


,target,readiness,notes
0,next_match_runs,READY FOR INTEGRATION,Required reports and artifacts passed the note...
1,next_match_wickets,READY FOR INTEGRATION,Required reports and artifacts passed the note...
2,next_match_runs_conceded,READY FOR INTEGRATION,Required reports and artifacts passed the note...
